In [ ]:
# Notebook 8/8 (Database Generation) - Database Pre-Processing
#
# Purpose:
# - Prepare the consolidated database for reproducible analysis and modelling.
# - Apply conservative text cleaning and standardization while preserving the schema expected
#   by the analysis notebook (Workbook 9).
#
# What this notebook does:
# 1) Loads the consolidated raw database created in prior steps.
# 2) Identifies the text/section columns and the set of columns required by the analysis notebook.
# 3) Applies deterministic, conservative text cleaning rules to section fields.
# 4) Produces:
#    - an analysis-compatible preprocessed dataset (same column names expected by Workbook 9)
#    - a modelling-only export with additional conservative exclusions
# 5) Produces a preprocessing report (rules applied, samples, and audit findings) for transparency.
#
# Inputs (Google Drive):
# - RAW_DB_IN: consolidated database CSV (test + control) used as input to preprocessing
# - ANALYSIS_NB: the analysis notebook used to detect required columns for downstream execution
#
# Outputs (Google Drive):
# - PREPROCESSED_OUT: analysis-compatible dataset written to OUT_DIR
# - MODELLING_OUT (if configured): modelling-only dataset with additional exclusions
# - Report files (CSV + Markdown) documenting rules and audit checks
#
# Execution:
# - Run top-to-bottom.
#
# Notes:
# This notebook is part of a larger, multi-stage data acquisition and
# analysis pipeline and is designed to be fully reproducible.

In [ ]:
# 1) Mount Drive + Paths

from google.colab import drive
drive.mount("/content/drive")

import os, re, json, time, glob, random
import pandas as pd
import numpy as np
from collections import defaultdict, Counter
from datetime import datetime, timezone

RAW_DB_IN   = "/content/drive/MyDrive/Dissertation/raw_db_final.csv"
ANALYSIS_NB = "/content/drive/MyDrive/Dissertation/9. READY TO RUN_Full_Analysis.ipynb"

OUT_DIR = "/content/drive/MyDrive/Dissertation/preprocessing_final"
os.makedirs(OUT_DIR, exist_ok=True)

PREPROCESSED_OUT      = os.path.join(OUT_DIR, "db_preprocessed.csv")
REPORT_RULES_CSV      = os.path.join(OUT_DIR, "preprocess_report_rules.csv")
REPORT_SAMPLES_CSV    = os.path.join(OUT_DIR, "preprocess_report_samples.csv")
REPORT_AUDIT_CSV      = os.path.join(OUT_DIR, "preprocess_audit_findings.csv")
REPORT_MD             = os.path.join(OUT_DIR, "preprocess_report.md")

MODEL_OUT             = os.path.join(OUT_DIR, "db_preprocessed_for_modelling.csv")
EXCLUDED_OUT          = os.path.join(OUT_DIR, "db_preprocessed_excluded.csv")

print("RAW_DB_IN:", RAW_DB_IN)
print("ANALYSIS_NB:", ANALYSIS_NB)
print("OUT_DIR:", OUT_DIR)

In [ ]:
# 2) Load Raw DB + Identify section/ text columns

db = pd.read_csv(RAW_DB_IN)
print("Rows, Cols:", db.shape)

print("\nFirst 40 columns:")
print(list(db.columns)[:40])

section_cols = [c for c in db.columns if c.startswith("section_") and c != "section_map_version"]
print("\nSection cols:", section_cols)

In [ ]:
# 3) Auto-detect “columns used in analysis” from the analysis notebook

with open(ANALYSIS_NB, "r", encoding="utf-8") as f:
    nb = json.load(f)

code_text = "\n".join(
    "".join(cell.get("source", "")) for cell in nb.get("cells", [])
    if cell.get("cell_type") == "code"
)

used_cols = sorted({c for c in db.columns if c in code_text})
print("Detected columns used in analysis:", len(used_cols))
print(used_cols[:80], "..." if len(used_cols) > 80 else "")

In [ ]:
# 4) Define columns to keep

def col_exists(c):
    return c in db.columns

# Strong identifiers
IDENTIFIERS = [c for c in [
    "paper_id",
    "base_id",
    "case_code",
    "case_code_norm",
    "class_label",
    "class_label_norm",
    "year",
    "source_pdf_path",
    "pdf_sha1",
] if col_exists(c)]

KEEP = sorted(set(IDENTIFIERS + used_cols))

# Keep all extracted sections
for c in section_cols:
    KEEP.append(c)
KEEP = sorted(set(KEEP))

print("KEEP columns:", len(KEEP))
db_keep = db[KEEP].copy()
print("db_keep shape:", db_keep.shape)

In [ ]:
# 5) Text cleaning utilities (conservative)

def is_blank(x):
    if pd.isna(x):
        return True
    s = str(x).strip().lower()
    return s == "" or s in {"nan","none","null","n/a","na","unknown"}

# 5.1) Line boilerplate patterns (conservative)
BOILERPLATE_LINE_PATTERNS = [
    r"^\s*downloaded\s+from\b",
    r"^\s*copyright\b",
    r"^\s*©\s*\d{4}\b",
    r"^\s*all\s+rights\s+reserved\b",
    r"^\s*terms\s+of\s+use\b",
    r"^\s*springer\s+nature\b",
    r"^\s*elsevier\b",
    r"^\s*wiley\b",
    r"^\s*taylor\s*&\s*francis\b",
    r"^\s*oxford\s+university\s+press\b",
    r"^\s*cambridge\s+university\s+press\b",
    r"^\s*bioRxiv\s+preprint\b",
    r"^\s*medRxiv\s+preprint\b",
    r"^\s*arXiv\b",
    r"^\s*preprint\b.*doi\b",
    r"^\s*page\s+\d+(\s+of\s+\d+)?\s*$",
    r"^\s*\d+\s*$",
    r"^\s*http[s]?://\S+\s*$",
    r"^\s*www\.\S+\s*$",
]
BOILERPLATE_RE = re.compile("|".join(f"(?:{p})" for p in BOILERPLATE_LINE_PATTERNS), re.IGNORECASE)

# 5.2) Leading tags to remove
LEADING_TAGS_RE = re.compile(
    r"^\s*(\[(retracted|withdrawn|retraction|erratum|corrigendum)\]\s*)+",
    re.IGNORECASE
)

# 5.3) Repeated junk prefix like "paper paper paper ..." at start
REPEATED_PAPER_RE = re.compile(r"^\s*(paper[\s\-\_:]*){2,}", re.IGNORECASE)

# 5.4) NEW: Artifact lines
LEADER_RE = re.compile(r"^(\.{8,}|-{8,}|_{8,}|={8,}|\*{6,}|·{6,}|•{6,})\s*$")
TOCISH_RE = re.compile(r"^[A-Za-z][A-Za-z\s\-]{2,60}\s+\.{6,}\s+\d{1,4}\s*$")

def normalize_whitespace(text: str) -> str:
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]{2,}", " ", text)
    return text.strip()

def drop_boilerplate_lines(text: str):
    lines = text.split("\n")
    kept = []
    removed = 0
    for ln in lines:
        stripped = ln.strip()

        # remove short junk punctuation lines
        if stripped and len(stripped) <= 3 and re.fullmatch(r"[\W\d_]+", stripped):
            removed += 1
            continue

        if BOILERPLATE_RE.search(stripped):
            removed += 1
            continue

        kept.append(ln)
    return "\n".join(kept).strip(), removed

def is_mostly_punct(line: str, min_len=30, punct_ratio=0.70):
    ln = line.strip()
    if len(ln) < min_len:
        return False
    punct = sum(1 for ch in ln if not ch.isalnum() and not ch.isspace())
    return (punct / max(1, len(ln))) >= punct_ratio

def drop_artifact_lines(text: str):
    """
    Conservative: remove only lines with strong evidence of extraction artifacts.
    - leader lines: ......., _____, =====
    - TOC-ish: "Heading ...... 12"
    - mostly punctuation blocks (long)
    """
    counters = Counter()
    if is_blank(text):
        return "", counters

    lines = text.replace("\r\n","\n").replace("\r","\n").split("\n")
    kept = []

    for ln in lines:
        stripped = ln.strip()
        if not stripped:
            kept.append(ln)  # preserve paragraph breaks
            continue

        if LEADER_RE.match(stripped):
            counters["rm_leader_lines"] += 1
            continue

        if TOCISH_RE.match(stripped):
            counters["rm_tocish_lines"] += 1
            continue

        if is_mostly_punct(stripped):
            counters["rm_mostly_punct_lines"] += 1
            continue

        kept.append(ln)

    return "\n".join(kept).strip(), counters

def clean_text(text: str):
    """
    Returns cleaned_text, actions(Counter)
    Conservative: only remove obvious non-content artifacts.
    """
    actions = Counter()
    if is_blank(text):
        return "", actions

    original = str(text)
    t = normalize_whitespace(original)

    # remove leading tags like [retracted]
    t2 = LEADING_TAGS_RE.sub("", t)
    if t2 != t:
        actions["rm_leading_tags"] += 1
        actions["chars_removed"] += (len(t) - len(t2))
        t = t2.strip()

    # remove repeated "paper" prefix at start
    t2 = REPEATED_PAPER_RE.sub("", t)
    if t2 != t:
        actions["rm_repeated_paper_prefix"] += 1
        actions["chars_removed"] += (len(t) - len(t2))
        t = t2.strip()

    # drop boilerplate lines (headers/footers/urls/page numbers)
    t2, n_removed_lines = drop_boilerplate_lines(t)
    if n_removed_lines > 0:
        actions["rm_boilerplate_lines"] += n_removed_lines
        actions["chars_removed"] += max(0, len(t) - len(t2))
        t = t2

    # drop artifact lines (leaders, TOC-ish, punctuation blocks)
    t2, c2 = drop_artifact_lines(t)
    if c2:
        for k, v in c2.items():
            actions[k] += v
        actions["chars_removed"] += max(0, len(t) - len(t2))
        t = t2

    # final normalization
    t2 = normalize_whitespace(t)
    if t2 != t:
        actions["norm_whitespace"] += 1
        actions["chars_removed"] += max(0, len(t) - len(t2))
        t = t2

    return t, actions

In [ ]:
# 6) Apply cleaning to text fields + create text_full_clean + rule reporting

# Sections to include in the model input
TEXT_SECTION_ORDER = [
    "section_title", "section_abstract", "section_introduction",
    "section_methods", "section_results", "section_discussion"
]
TEXT_SECTION_ORDER = [c for c in TEXT_SECTION_ORDER if c in db_keep.columns]

TEXT_COLS = [c for c in TEXT_SECTION_ORDER if c in db_keep.columns]
print("Text columns to clean:", TEXT_COLS)

rule_agg = Counter()
samples = []
MAX_SAMPLES_PER_RULE = 5
samples_per_rule = defaultdict(int)

def add_sample(rule_name, paper_id, col, before, after):
    if samples_per_rule[rule_name] >= MAX_SAMPLES_PER_RULE:
        return
    samples_per_rule[rule_name] += 1
    samples.append({
        "rule": rule_name,
        "paper_id": paper_id,
        "column": col,
        "before_snippet": (before[:500] + "…") if len(before) > 500 else before,
        "after_snippet":  (after[:500] + "…") if len(after)  > 500 else after,
    })

docs_modified = set()

for col in TEXT_COLS:
    clean_col = f"{col}_clean"
    cleaned = []

    for idx, row in db_keep.iterrows():
        before = "" if is_blank(row.get(col, "")) else str(row[col])
        after, actions = clean_text(before)
        cleaned.append(after)

        if before.strip() != after.strip():
            docs_modified.add(str(row.get("paper_id", idx)))

        for k, v in actions.items():
            rule_agg[k] += v
            if k not in {"chars_removed"} and before.strip() != after.strip():
                add_sample(k, row.get("paper_id", idx), col, before, after)

    db_keep[clean_col] = cleaned
    print(f"Cleaned {col} -> {clean_col}")

def concat_sections(row, cols):
    parts = []
    for c in cols:
        v = row.get(f"{c}_clean", "")
        if not is_blank(v):
            parts.append(str(v).strip())
    return "\n\n".join(parts).strip()

db_keep["text_full_clean"] = db_keep.apply(lambda r: concat_sections(r, TEXT_COLS), axis=1)

print("\nCleaning done.")
print("Docs modified (at least one change):", len(docs_modified))
print("Top rules:", rule_agg.most_common(10))

In [ ]:
# 7) Corpus audit (identify other problems + examples)

DOT_RUN_RE = re.compile(r"\.{10,}")
PUNCT_ONLY_RE = re.compile(r"^[\W_]+$")
PAGE_NUM_LINE_RE = re.compile(r"^\s*(page\s*)?\d{1,4}(\s+of\s+\d{1,4})?\s*$", re.IGNORECASE)

def iter_lines(x):
    if is_blank(x):
        return []
    s = str(x).replace("\r\n", "\n").replace("\r", "\n")
    return [ln.strip() for ln in s.split("\n") if ln.strip()]

audit_counts = Counter()
audit_examples = defaultdict(list)

# Scan CLEANED combined text (most relevant to modelling)
for text in db_keep["text_full_clean"].fillna("").astype(str).values:
    if not text.strip():
        continue
    for ln in iter_lines(text):
        if DOT_RUN_RE.search(ln):
            audit_counts["dot_runs_present"] += 1
            if len(audit_examples["dot_runs_present"]) < 10: audit_examples["dot_runs_present"].append(ln[:200])
        if PUNCT_ONLY_RE.match(ln) and len(ln) >= 12:
            audit_counts["punct_only_present"] += 1
            if len(audit_examples["punct_only_present"]) < 10: audit_examples["punct_only_present"].append(ln[:200])
        if PAGE_NUM_LINE_RE.match(ln):
            audit_counts["page_num_line_present"] += 1
            if len(audit_examples["page_num_line_present"]) < 10: audit_examples["page_num_line_present"].append(ln[:200])
        if TOCISH_RE.match(ln):
            audit_counts["tocish_present"] += 1
            if len(audit_examples["tocish_present"]) < 10: audit_examples["tocish_present"].append(ln[:200])

print("Audit counts (in text_full_clean):")
for k,v in audit_counts.most_common():
    print(f"{k:>22}: {v}")

print("\nExamples:")
for k, exs in audit_examples.items():
    print(f"\n--- {k} ---")
    for e in exs[:5]:
        print(" ", e)

# Title-only/ short-content docs (based on cleaned sections excluding title)
non_title_clean_cols = [f"{c}_clean" for c in TEXT_COLS if c != "section_title"]
def total_non_title_len(row):
    return sum(len(str(row.get(c,""))) for c in non_title_clean_cols if not is_blank(row.get(c,"")))

db_keep["_non_title_chars_clean"] = db_keep.apply(total_non_title_len, axis=1)
db_keep["_has_title_clean"] = db_keep["section_title_clean"].fillna("").astype(str).str.strip().ne("") if "section_title_clean" in db_keep.columns else False

title_only_mask = db_keep["_has_title_clean"] & (db_keep["_non_title_chars_clean"] < 80)
short_doc_mask  = (db_keep["_non_title_chars_clean"] > 0) & (db_keep["_non_title_chars_clean"] < 400)

print("\nDoc-level flags (cleaned):")
print("title-only docs:", int(title_only_mask.sum()))
print("short docs (<400 chars excluding title):", int(short_doc_mask.sum()))

# Save audit findings
audit_rows = [{"metric": k, "count": int(v)} for k,v in audit_counts.items()]
audit_rows += [
    {"metric": "title_only_docs", "count": int(title_only_mask.sum())},
    {"metric": "short_docs_excl_title_lt_400", "count": int(short_doc_mask.sum())},
]
pd.DataFrame(audit_rows).sort_values("count", ascending=False).to_csv(REPORT_AUDIT_CSV, index=False)
print("\nSaved audit findings:", REPORT_AUDIT_CSV)

In [ ]:
# 8) Final preprocessed dataset (NO _clean columns; analysis-compatible; includes text_full for QC/exclusions)

# 1) Overwrite original section columns with cleaned versions (so names match analysis)
for c in TEXT_COLS:
    cc = f"{c}_clean"
    if cc in db_keep.columns:
        db_keep[c] = db_keep[cc]

# 2) Build combined cleaned text WITHOUT using _clean naming
db_keep["text_full"] = db_keep.apply(
    lambda r: "\n\n".join(
        str(r[c]).strip()
        for c in TEXT_COLS
        if c in db_keep.columns and str(r[c]).strip()
    ).strip(),
    axis=1
)

# 3) Columns required by the analysis notebook
REQUIRED_FOR_ANALYSIS = [
    "paper_id",
    "class_label",
    "quality_percent",
    "applicability",
    "section_title",
    "section_abstract",
    "section_introduction",
    "section_methods",
    "section_results",
    "section_discussion",
]

# Optional (keep if present)
OPTIONAL = [c for c in [
    "doi_norm",
    "base_id",
    "case_code_norm",
    "class_label_norm",
    "year",
] if c in db_keep.columns]

# Always include text_full
FINAL_COLS = [c for c in REQUIRED_FOR_ANALYSIS if c in db_keep.columns] + ["text_full"] + OPTIONAL

missing = [c for c in REQUIRED_FOR_ANALYSIS if c not in db_keep.columns]
if missing:
    raise ValueError(f"Missing columns required by analysis: {missing}")

db_pre = db_keep[FINAL_COLS].copy()

# 4) Ensure no _clean columns are exported
db_pre = db_pre.drop(columns=[c for c in db_pre.columns if c.endswith("_clean")], errors="ignore")

print("Final preprocessed columns:", list(db_pre.columns))
print("Rows:", len(db_pre), "Cols:", db_pre.shape[1])

db_pre.to_csv(PREPROCESSED_OUT, index=False)
print("Saved:", PREPROCESSED_OUT)

In [ ]:
# 9) Modelling export only: conservative exclusions + STRICT non-Latin + keep only paired test+control

# 9.1) Thresholds
MIN_MODEL_CHARS = 800
MIN_ALPHA_RATIO = 0.45

# STRICT: exclude if ANY non-Latin script char exists
RX_CYRILLIC   = re.compile(r"[\u0400-\u04FF]")
RX_CJK        = re.compile(r"[\u4E00-\u9FFF]")   # Han (Chinese/Japanese Kanji)
RX_HANGUL     = re.compile(r"[\uAC00-\uD7AF]")
RX_ARABIC     = re.compile(r"[\u0600-\u06FF]")
RX_HEBREW     = re.compile(r"[\u0590-\u05FF]")
RX_DEVANAGARI = re.compile(r"[\u0900-\u097F]")

SCRIPT_PATTERNS = {
    "cyrillic": RX_CYRILLIC,
    "cjk": RX_CJK,
    "hangul": RX_HANGUL,
    "arabic": RX_ARABIC,
    "hebrew": RX_HEBREW,
    "devanagari": RX_DEVANAGARI,
}

def is_blank(x):
    if pd.isna(x):
        return True
    s = str(x).strip().lower()
    return s == "" or s in {"nan", "none", "null", "n/a", "na", "unknown"}

def alpha_ratio(s):
    """Ratio of unicode letters over total chars."""
    if is_blank(s):
        return 0.0
    s = str(s)
    alpha = sum(1 for ch in s if ch.isalpha())
    return alpha / max(1, len(s))

def first_nonlatin_script(s):
    """Return script name if ANY non-Latin script char is present, else ''."""
    if is_blank(s):
        return ""
    s = str(s)
    for name, rx in SCRIPT_PATTERNS.items():
        if rx.search(s):
            return name
    return ""

def normalize_label(s):
    """Normalize label to 'test' or 'control' if possible."""
    if pd.isna(s):
        return ""
    t = str(s).strip().lower()
    # common variants
    if t in {"test", "case", "retracted", "positive", "1", "true", "yes", "y"}:
        return "test"
    if t in {"control", "non-retracted", "nonretracted", "negative", "0", "false", "no", "n"}:
        return "control"
    return t  # keep as-is for debugging if unexpected

# 9.2) Load preprocessed DB
db_pre = pd.read_csv(PREPROCESSED_OUT)

# Required columns for this step
for req in ["text_full", "base_id", "class_label"]:
    if req not in db_pre.columns:
        raise ValueError(f"Required column missing: '{req}'")

# Prefer class_label_norm if present, else class_label
LABEL_SRC = "class_label_norm" if "class_label_norm" in db_pre.columns else "class_label"

# 9.3) Compute exclusion metrics
db_pre["_nchars"] = db_pre["text_full"].fillna("").astype(str).str.len()
db_pre["_alpha_ratio"] = db_pre["text_full"].apply(alpha_ratio)
db_pre["_nonlatin_script"] = db_pre["text_full"].apply(first_nonlatin_script)

m_short = db_pre["_nchars"] < MIN_MODEL_CHARS
m_low_alpha = (db_pre["_nchars"] >= MIN_MODEL_CHARS) & (db_pre["_alpha_ratio"] < MIN_ALPHA_RATIO)
m_nonlatin = db_pre["_nonlatin_script"].ne("")   # STRICT: any hit excludes

# Apply these exclusions ONLY for modelling export
keep_mask_rules = ~(m_short | m_low_alpha | m_nonlatin)

# 9.4) Create candidate modelling set (rule-passing)
db_candidate = db_pre.loc[keep_mask_rules].copy()

# Normalize labels to test/control for pairing logic
db_candidate["_label_norm_for_pairing"] = db_candidate[LABEL_SRC].apply(normalize_label)

# 9.5) Pairing requirement: keep only base_id that has BOTH test + control
# -----------------------------
pair_sets = db_candidate.groupby("base_id")["_label_norm_for_pairing"].agg(lambda s: set(s))
paired_base_ids = set(pair_sets[pair_sets.apply(lambda s: ("test" in s) and ("control" in s))].index.astype(str))

m_paired = db_candidate["base_id"].astype(str).isin(paired_base_ids)

db_model = db_candidate.loc[m_paired].copy()
db_unpaired = db_candidate.loc[~m_paired].copy()

# 9.6. Build excluded audit with reasons
db_excl_rules = db_pre.loc[~keep_mask_rules].copy()

def build_reason_row(row):
    reasons = []
    if row.get("_nchars", 0) < MIN_MODEL_CHARS:
        reasons.append("too_short")
    if (row.get("_nchars", 0) >= MIN_MODEL_CHARS) and (row.get("_alpha_ratio", 1.0) < MIN_ALPHA_RATIO):
        reasons.append("low_alpha")
    if str(row.get("_nonlatin_script", "")).strip() != "":
        reasons.append(f"nonlatin:{row.get('_nonlatin_script')}")
    return "|".join(reasons) if reasons else "excluded"

db_excl_rules["_excl_reason"] = db_excl_rules.apply(build_reason_row, axis=1)
db_unpaired["_excl_reason"] = "unpaired_missing_test_or_control"

db_excluded = pd.concat([db_excl_rules, db_unpaired], ignore_index=True)

# Drop helper columns from modelling output (keep them in excluded audit if you want)
drop_helpers_model = ["_nchars", "_alpha_ratio", "_nonlatin_script", "_label_norm_for_pairing"]
db_model = db_model.drop(columns=drop_helpers_model, errors="ignore")

# 9.7) Save outputs
db_model.to_csv(MODEL_OUT, index=False)
db_excluded.to_csv(EXCLUDED_OUT, index=False)

print("Exclusions (rule-based):")
print("  too_short:", int(m_short.sum()))
print("  low_alpha_ratio:", int(m_low_alpha.sum()))
print("  non_latin_any:", int(m_nonlatin.sum()))
print("\nPairing filter:")
print("  candidate_after_rules:", len(db_candidate))
print("  excluded_unpaired:", len(db_unpaired))
print("  kept_paired_for_modelling:", len(db_model))

print("\nSaved modelling dataset:", MODEL_OUT)
print("Saved excluded audit:", EXCLUDED_OUT)

# 9.8) Verification checks
# 1) No non-Latin scripts remain in modelling set
nonlatin_left = db_model["text_full"].fillna("").astype(str).apply(
    lambda s: any(rx.search(s) for rx in SCRIPT_PATTERNS.values())
).sum()
print("Non-Latin docs remaining in modelling dataset:", int(nonlatin_left))

# 2) Ensure pairing holds: every base_id has both test and control
lbl_check = db_model[LABEL_SRC].apply(normalize_label)
pair_check = db_model.assign(_lbl=lbl_check).groupby("base_id")["_lbl"].agg(lambda s: set(s))
bad_pairs = pair_check[~pair_check.apply(lambda s: ("test" in s) and ("control" in s))]
print("Base_ids failing pairing in modelling dataset:", int(len(bad_pairs)))

In [ ]:
# 10) Full preprocessing report (rules + samples + markdown)

# Rules summary
rules_df = pd.DataFrame(
    [{"rule": "docs_modified", "count": int(len(docs_modified))}]
)
rules_df = pd.concat(
    [
        rules_df,
        pd.DataFrame(
            [{"rule": k, "count": int(v)} for k, v in rule_agg.items()]
        ),
    ],
    ignore_index=True,
).sort_values("count", ascending=False)

rules_df.to_csv(REPORT_RULES_CSV, index=False)
print("Saved rules report:", REPORT_RULES_CSV)

# Samples report
samples_df = pd.DataFrame(samples)
samples_df.to_csv(REPORT_SAMPLES_CSV, index=False)
print("Saved samples report:", REPORT_SAMPLES_CSV)

# Reload final preprocessed dataset to list columns safely
db_pre = pd.read_csv(PREPROCESSED_OUT)

# Markdown report
ts = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")

md = []
md.append(f"# Preprocessing Report\n\nGenerated: **{ts}**\n")

md.append("## Inputs / Outputs\n")
md.append(f"- Raw DB: `{RAW_DB_IN}`\n")
md.append(f"- Analysis notebook: `{ANALYSIS_NB}`\n")
md.append(f"- Preprocessed DB: `{PREPROCESSED_OUT}`\n")

if os.path.exists(MODEL_OUT):
    md.append(f"- Modelling DB (optional): `{MODEL_OUT}`\n")
if os.path.exists(EXCLUDED_OUT):
    md.append(f"- Excluded DB (optional): `{EXCLUDED_OUT}`\n")

md.append("\n## Dataset sizes\n")
md.append(f"- Raw rows: **{len(db)}**\n")
md.append(f"- Preprocessed rows: **{len(db_pre)}**\n")

md.append("\n## Columns in preprocessed dataset\n")
md.append(", ".join(db_pre.columns) + "\n")

md.append("\n## Cleaning rules — counts\n")
md.append(rules_df.to_markdown(index=False))

md.append("\n\n## Samples (snippets)\n")
if len(samples_df) == 0:
    md.append("_No changes were applied._\n")
else:
    md.append(samples_df.head(40).to_markdown(index=False))

with open(REPORT_MD, "w", encoding="utf-8") as f:
    f.write("\n".join(md))

print("Saved markdown report:", REPORT_MD)